In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# I. KHAI PHÁ (EDA) VÀ TIỀN XỬ LÝ DỮ LIỆU

## 1.1. Khai phá dữ liệu (EDA)

In [ ]:
# Tổng quan dữ liệu: describe, info
# Kiểm tra dữ liệu: missing, duplicate
# Vissualization để trực quan hơn
# Bảng thống kê mô tả (mean, median, std, min, max) các feature numeric
# Histogram phân phối từng feature numeric
# Boxplot từng feature numeric
# Bar chart tần suất từng feature categorical
# Biểu đồ phân phối target y (yes/no)

#-------------------------------------------------------------------------------------------------#
#import dữ liệu
df=pd.read_csv("/kaggle/input/datasets/vchuhoiphong/dataset-dm/bank.csv",sep=";")
df.info()
df.describe()

#tinh so mising trong du lieu
missing_count= df.isnull().sum()
#phan tram trong trong du lieu
missing_rate= 100 * df.isnull().mean() 
#trùng lặp dữ liệu
duplicated = df.duplicated().sum()

#total trong trong du lieu
total_missing = df.isnull().sum().sum()
total_value = df.shape[0]*df.shape[1]

missing_rate_total= 100*(total_missing/total_value)

print("So missing tung thanh phan la: \n",missing_count)
print("\n Ty le missing la: \n",missing_rate)
print ("\n Tong so missing cua data:\n", missing_rate_total)
print ("\n Tong so duplicated cua data:\n", duplicated)

#Thông tin trong côt dữ liệu y 
print(df['y'].value_counts())
print(df['y'].value_counts(normalize=True) * 100)


**Nhận xét**
* Dữ liệu có khoảng 45.000 dòng, với 16 cột dữ liệu (bao gồm 1 cột "y" dự đoán) và 2 loại kiểu chính là int và object
* Tỷ lệ duplicate và missing là 0% => Không cần điền khuyết và lọc bỏ trùng lặp
* Tuy nhiên, cột "y" có tỷ lệ "yes" chỉ khoảng 12% trong khi tỉ lệ "no" đạt khoảng 88% => mất cân bằng nghiêm trọng, cần xử lý trước khi huấn luyện
  

In [ ]:
#Visuallization
num_cols= df.select_dtypes(include=np.number).columns.tolist()
cat_cols= df.select_dtypes(exclude=np.number).columns.drop("y").tolist() #cat=categorical

#Hiển thị thông tin về thống kê của số
print("------------------------Chỉ số thống kê của các featrue------------------------")
statstic = df[num_cols].agg(["mean","median","std","min","max"])
print(statstic)

#Historgram của các feature
print("------------------------Historgram của các feature------------------------")
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, c in enumerate(num_cols):
    sns.histplot(df[c], bins=30, kde=True, ax=axes[i])
    axes[i].set_title(c)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

#Boxplot của các feature
print("------------------------Boxplot của các feature------------------------")
for i, c in enumerate(num_cols):
    sns.boxplot(x=df[c], ax=axes[i])
    axes[i].set_title(c)

plt.tight_layout()
plt.show()

#categorical
print("------------------------Historgram của các categorical------------------------")
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for i, c in enumerate(cat_cols):
    sns.countplot(data=df, x=c, ax=axes[i])
    axes[i].set_title(c)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

#Phân phối của y
print("------------------------Historgram của các 'y'------------------------")
plt.figure(figsize=(5, 4))
ax = sns.countplot(data=df, x='y')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.title("Phan phoi target y")
plt.show()

**Nhận xét**
* Hầu hết các cột dữ liệu bị lệch phải
* Phạm vi về độ lớn dữ liệu khác nhau => thực hiện scaling cho tiền xử lý
* Có giá trị âm ở cột balance và pday, tuy nhiên nó hợp lý do mang ý nghĩa là nợ xấu và chưa được gọi điện
* Cột duration-thời gian gọi điện, nếu 0 thì chắc chắn 'no' => Chủ tập dataset khuyên bỏ do vấn đề data leakage

## 1.2. Tiền xử lý dữ liệu

### 1.2.1 Data Cleaning

In [ ]:
# Bảng số lượng outlier bị loại theo từng cột
# Boxplot so sánh trước/sau xử lý outlier
# Bảng xử lý missing/unknown (cột nào, phương pháp nào)
# Số liệu tổng số dòng trước/sau cleaning

#code

### 1.2.2 Data Transformation

In [ ]:
# Bảng mapping encoding (categorical → số cột mới sinh ra)
# Shape dữ liệu trước/sau transform (số dòng × số cột)
# Histogram so sánh trước/sau scaling (1-2 feature minh họa)
# Bảng thống kê mean/std sau scaling

#code

### 1.2.3 Xử lý Imbalance

In [ ]:
# Bar chart phân phối class trước/sau SMOTE (train only)
# Bảng số liệu số sample mỗi class trước/sau
# Scatter plot PCA/t-SNE so sánh phân bố dữ liệu (optional)
# Xác nhận bằng số liệu: val/test không bị SMOTE

#code

### 1.2.4 Lựa chọn đặc trưng (feature selection)

In [ ]:
# Bar chart feature importance (sắp xếp giảm dần)
# Bảng số liệu importance score từng feature
# Cumulative importance plot
# Bảng so sánh số lượng feature trước/sau selection
# Bảng so sánh hiệu suất model full feature vs feature đã chọn (optional)

#code

### 1.2.5 Chia tập dữ liệu

In [ ]:
#code

# II. HUẤN LUYỆN MÔ HÌNH


In [ ]:
#YÊU CẦU CHUNG

# Bảng kết quả Precision, Recall, F1-score, ROC-AUC, PR-AUC (trên tập test)
# Confusion Matrix (dạng heatmap)
# ROC Curve (đường cong + giá trị AUC)
# PR Curve (Precision-Recall Curve + giá trị AUC)
# Classification Report đầy đủ (per-class: precision/recall/f1 cho cả "yes" và "no")

# 2.1. Mô hình học máy truyền thống

### 2.1.1 Logistic Regression

In [ ]:
#code

### 2.1.2 Gradient Boosting

In [ ]:
#code

# 2.2. Mô hình học sâu

### 2.2.1 1D_CNN

In [ ]:
# Learning curve (loss/accuracy qua từng epoch, train vs validation)
# Bảng kiến trúc mô hình (số layer, kernel size, số filter)

#code

### 2.2.2 Tab transformer

In [ ]:
#Learning curve (loss/accuracy qua từng epoch, train vs validation)
# Bảng kiến trúc mô hình (số attention head, số transformer block, embedding dimension)

#code

### 2.2.3 FT-Transformer

In [ ]:
#Learning curve (loss/accuracy qua từng epoch, train vs validation)
# Bảng kiến trúc mô hình (số attention head, số transformer block, embedding dimension)

#code